In [1]:
import pandas as pd
df_train = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Train/MINT/FewRel_train_phrase_preprocess.csv')
print(df_train.shape) 
df_train.head(2)

(4000, 15)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,subj_id,obj_id,token,gpt4_phrase,gpt4_phrase_preprocessed
0,train_P1877_557,after a work by,P1877,"After reading Alan Glynn 's "" The Dark Fields ...",limitless,alan glynn,14,14,2,3,Q19911394,Q2830507,"['After', 'reading', 'Alan', 'Glynn', ""'s"", '""...",inspired_by_author,inspired_by_author
1,train_P1877_158,after a work by,P1877,"It was a sequel to the 1952 film "" Heidi "" , w...",heidi,johanna spyri,9,9,25,26,Q455508,Q123053,"['It', 'was', 'a', 'sequel', 'to', 'the', '195...",adapted_from_author,adapted_from_author


In [2]:
df_train['gpt4_phrase'].nunique()

1813

In [3]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize


texts = df_train["gpt4_phrase"].fillna("").astype(str).tolist()



print("num texts:", len(texts))


model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("embedding shape:", embeddings.shape)


def estimate_k_by_zhang2021(embeddings, k0, random_state=42):
    """
    Zhang et al. (2021a) 방식의 unknown K 추정.

    Args:
        embeddings   : clustering할 embedding matrix, shape = (N, dim)
        k0           : over-clustering에 사용할 큰 cluster 개수
        random_state : random seed

    Returns:
        estimated_k  : 추정된 cluster 개수
        labels_k0    : K0로 clustering한 결과
        cluster_info : cluster별 size 정보
    """

    n = len(embeddings)

    kmeans = KMeans(
        n_clusters=k0,
        random_state=random_state,
        n_init="auto"
    )

    labels_k0 = kmeans.fit_predict(embeddings)

    cluster_sizes = pd.Series(labels_k0).value_counts().sort_index()

    threshold = n / k0

    valid_clusters = cluster_sizes[cluster_sizes >= threshold].index.tolist()
    estimated_k = len(valid_clusters)

    cluster_info = pd.DataFrame({
        "cluster_id": cluster_sizes.index,
        "size": cluster_sizes.values,
        "threshold": threshold,
        "is_valid": cluster_sizes.values >= threshold
    })

    return estimated_k, labels_k0, cluster_info

num texts: 4000


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

embedding shape: (4000, 384)


In [4]:
k0 = 300

estimated_k, labels_k0, cluster_info = estimate_k_by_zhang2021(
    embeddings=embeddings,
    k0=k0,
    random_state=42
)

print("Estimated K:", estimated_k)
cluster_info

Estimated K: 96


,cluster_id,size,threshold,is_valid
0,0,15,13.333333,True
1,1,27,13.333333,True
2,2,214,13.333333,True
3,3,1,13.333333,False
4,4,18,13.333333,True
...,...,...,...,...
295,295,4,13.333333,False
296,296,1,13.333333,False
297,297,4,13.333333,False
298,298,11,13.333333,False


In [5]:
kmeans = KMeans(
    n_clusters=estimated_k,
    random_state=42,
    n_init="auto"
)

df_train["cluster_unknown_k"] = kmeans.fit_predict(embeddings)


# Evaluation

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    v_measure_score,
    homogeneity_score,
    completeness_score,
    confusion_matrix
)
from scipy.optimize import linear_sum_assignment


def evaluate_clustering(df: pd.DataFrame, pred_col: str, gold_col: str = "relation"):

    
    non_na_df = df[~df[pred_col].isna()].copy()

    if len(non_na_df) == 0:
        raise ValueError(f"{pred_col}에 유효한 값이 없습니다.")

    y_true = non_na_df[gold_col].astype(str).values
    y_pred = non_na_df[pred_col].astype(str).values

    
    overall_purity = (
        non_na_df
        .groupby(pred_col)[gold_col]
        .value_counts()
        .groupby(level=0)
        .apply(lambda x: x.max())
        .sum()
        / len(non_na_df)
    )

    
    ari = adjusted_rand_score(y_true, y_pred)

    hom = homogeneity_score(y_true, y_pred)
    com = completeness_score(y_true, y_pred)
    v_f1 = v_measure_score(y_true, y_pred)

    
    def b3_precision_recall_f1(y_true, y_pred):

        true_groups = {}
        pred_groups = {}

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            true_groups.setdefault(t, set()).add(i)
            pred_groups.setdefault(p, set()).add(i)

        P_sum, R_sum = 0.0, 0.0
        n = len(y_true)

        for i, (t, p) in enumerate(zip(y_true, y_pred)):
            T = true_groups[t]
            C = pred_groups[p]
            inter = len(T & C)

            P_sum += inter / len(C) if len(C) > 0 else 0.0
            R_sum += inter / len(T) if len(T) > 0 else 0.0

        P = P_sum / n
        R = R_sum / n
        F1 = (2 * P * R / (P + R)) if (P + R) > 0 else 0.0

        return P, R, F1

    b3_p, b3_r, b3_f1 = b3_precision_recall_f1(y_true, y_pred)

    
    cm = confusion_matrix(y_true, y_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)
    hungarian_acc = cm[row_ind, col_ind].sum() / cm.sum()

    
    print(f"Purity : {overall_purity:.4f}")
    print(f"ACC    : {hungarian_acc:.4f}")
    
    
    print(f"B3-F1  : {b3_f1:.4f}")
    
    
    print(f"V-F1   : {v_f1:.4f}")
    print(f"ARI    : {ari:.4f}")

    return {
        "Purity": overall_purity,
        "ARI": ari,
        "V-hom": hom,
        "V-com": com,
        "V-F1": v_f1,
        "B3-P": b3_p,
        "B3-R": b3_r,
        "B3-F1": b3_f1,
        "ACC": hungarian_acc,
    }

In [7]:


metrics = evaluate_clustering(
    df=df_train,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics

Purity : 0.5048
ACC    : 0.4500
B3-F1  : 0.3645
V-F1   : 0.6540
ARI    : 0.2909


{'Purity': 0.50475,
 'ARI': 0.29086448112683255,
 'V-hom': 0.6542433779649246,
 'V-com': 0.6537874638253457,
 'V-F1': 0.6540153414407003,
 'B3-P': 0.38071517955586703,
 'B3-R': 0.34965000000000057,
 'B3-F1': 0.364521930283306,
 'ACC': 0.45}

# Test set Evaluation

In [8]:
test_df = pd.read_csv('/workspace/KG-known0/ARR2026/Data_paper/FewRel/Test/FewRel_test_phrase.csv')
print(test_df.shape)
test_df.head(2)

(1600, 14)


,id,relation,relation_id,text,subject,object,subj_start,subj_end,obj_start,obj_end,subj_id,obj_id,token,gpt4_phrase
0,train_P1877_500,after a work by,P1877,"In 1998 , she starred opposite Elisabeth Shue ...",cousin bette,balzac,16,17,13,13,Q835676,Q9711,"['In', '1998', ',', 'she', 'starred', 'opposit...",work_author
1,train_P1877_396,after a work by,P1877,"He directed Disney 's "" Tom and Huck "" in 1995...",tom and huck,mark twain,5,7,15,16,Q1616185,Q7245,"['He', 'directed', 'Disney', ""'s"", '""', 'Tom',...",adaptation_author


In [9]:



if "gpt4_phrase" not in test_df.columns:
    raise ValueError("test_df에 'gpt4_phrase' 컬럼이 없습니다.")

if "relation" not in test_df.columns:
    raise ValueError("test_df에 gold label 컬럼 'relation'이 없습니다.")

test_texts = test_df["gpt4_phrase"].fillna("").astype(str).tolist()

print("num test instances:", len(test_texts))

test_embeddings = model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("test_embeddings shape:", test_embeddings.shape)

test_df["cluster_unknown_k"] = kmeans.predict(test_embeddings)

print(test_df[[
    "relation",
    "gpt4_phrase",
    "cluster_unknown_k"
]].head(10))

num test instances: 1600


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

test_embeddings shape: (1600, 384)
          relation                        gpt4_phrase  cluster_unknown_k
0  after a work by                        work_author                 39
1  after a work by                  adaptation_author                 35
2  after a work by                    work_authorship                 39
3  after a work by                    based_on_author                 39
4  after a work by                  adaptation_author                 35
5  after a work by                 inspiration_author                 39
6  after a work by                        work_author                 39
7  after a work by                  adaptation_author                 35
8  after a work by                        inspired_by                 16
9  after a work by  based_on_author_of_original_novel                 39


In [10]:
metrics_kmeans_test = evaluate_clustering(
    df=test_df,
    pred_col="cluster_unknown_k",
    gold_col="relation"
)

metrics_kmeans_test

Purity : 0.5062
ACC    : 0.4481
B3-F1  : 0.3884
V-F1   : 0.6924
ARI    : 0.2775


{'Purity': 0.50625,
 'ARI': 0.27745440331732657,
 'V-hom': 0.6886005165029938,
 'V-com': 0.6961701197678737,
 'V-F1': 0.6923646291880778,
 'B3-P': 0.39847577188418354,
 'B3-R': 0.3788124999999989,
 'B3-F1': 0.3883954223855016,
 'ACC': 0.448125}